# Notebook 01: Data Understanding
## Forecasting Education Performance -- Tanzania BEST Datasets (2020-2024)
**Author:** Habil Masawika | **Project:** Tanzania BEST ML Forecasting

---

### Objectives
This notebook establishes a comprehensive understanding of the raw Tanzania BEST (Basic Education
Statistics in Tanzania) datasets before any modelling begins. Good modelling starts with deep
data understanding. The goals here are to:

1. Load all five annual BEST Excel workbooks and enumerate their sheet structures
2. Identify the most analytically relevant sheets for secondary education performance
3. Preview raw table structures and understand extraction challenges
4. Produce a structured data inventory documenting availability across years
5. Lay the groundwork for the extraction and harmonisation pipeline in Notebook 02

### Background
The BEST reports are published annually by Tanzania Ministry of Education, Science and
Technology (MoEST). Each edition contains 150-190 Excel worksheets covering the entire national
education system from pre-primary through higher education. The secondary education section
(Section 3) is the focus of this project, as it contains data on the Certificate of Secondary
Education Examination (CSEE), enrolment, teacher deployment, infrastructure, and student flow rates.

A key challenge is that the 2020 edition uses a numbered flat-table format (Table N), while
2021-2024 use a named-sheet format (T3.*), requiring careful sheet mapping before any data
can be extracted.

In [ ]:
import sys, os, warnings
sys.path.insert(0, '/home/claude/BEST-ML-Forecasting/src')
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from utilities import setup_logging, set_seeds, get_environment_info, ProjectPaths, Timer
from data_loader import BESTLoader, SHEET_MAP, EXPECTED_REGIONS

set_seeds(42)
logger = setup_logging("INFO")
paths  = ProjectPaths()

print("Environment:")
for k, v in get_environment_info().items():
    print(f"  {k}: {v}")

In [ ]:
# ── Load all workbooks ────────────────────────────────────────────────────
with Timer("Workbook loading"):
    loader = BESTLoader(data_dir="/mnt/user-data/uploads")

print(f"\nWorkbooks loaded: {len(loader.workbooks)}")
for yr, xl in loader.workbooks.items():
    print(f"  {yr}: {len(xl.sheet_names):>3} sheets | "
          f"First 5: {xl.sheet_names[:5]}")

In [ ]:
# ── Sheet inventory ──────────────────────────────────────────────────────
inventory = loader.sheet_inventory()
print(f"Total unique sheets across all years: {len(inventory)}")
print("\nSample (T3.* secondary sheets):")
t3 = inventory[inventory['sheet'].str.startswith('T3')]
print(t3.head(30).to_string(index=False))

In [ ]:
# ── Map analytical sheets across years ───────────────────────────────────
print("Analytical sheet mapping (canonical name -> year availability):")
print()
headers = ["Canonical Name", "2020", "2021", "2022", "2023", "2024"]
print(f"{'Canonical Name':<30}" + "".join(f"{yr:>8}" for yr in [2020,2021,2022,2023,2024]))
print("-" * 70)
for canonical, year_map in SHEET_MAP.items():
    row = f"{canonical:<30}"
    for yr in [2020,2021,2022,2023,2024]:
        sheet = year_map.get(yr, "--")
        if sheet and sheet != "--":
            xl = loader.workbooks.get(yr)
            found = xl and any(s.strip() == sheet.strip() for s in xl.sheet_names)
            row += f"{'  Y  ':>8}" if found else f"{'  ?  ':>8}"
        else:
            row += f"{'  --  ':>8}"
    print(row)

In [ ]:
# ── Preview 5 key sheets ─────────────────────────────────────────────────
previews = [
    (2021, 'T3.25PassRateCSEE', 'CSEE National Pass Rate Trend'),
    (2021, 'T3.1#ofschools',    'Schools by Region'),
    (2021, 'T3.30Tot-QPTR&PTR', 'National PTR and Teacher Summary'),
    (2021, 'T3.39Tot-Electric', 'Electricity Access by Region'),
    (2021, 'T3.41Tot-ICTEquip', 'ICT Equipment by Region'),
]
for yr, sheet, label in previews:
    xl = loader.workbooks[yr]
    if sheet in xl.sheet_names:
        df = pd.read_excel(xl, sheet_name=sheet, header=None, nrows=8)
        print(f"\n{'='*60}")
        print(f"Sheet: {sheet}  ({label})")
        print('='*60)
        print(df.iloc[:, :8].to_string())

In [ ]:
# ── 2020 format: numbered tables ─────────────────────────────────────────
print("2020 file -- secondary-related tables (scan by title):")
print()
xl20 = loader.workbooks[2020]
for sname in xl20.sheet_names:
    try:
        df = pd.read_excel(xl20, sheet_name=sname, header=None, nrows=2)
        cell0 = str(df.iloc[0, 0])
        cell1 = str(df.iloc[1, 0]) if len(df) > 1 else ""
        if 'Secondary Education' in cell0:
            print(f"  {sname}: {cell1[:90]}")
    except Exception:
        pass

In [ ]:
# ── Region coverage check ────────────────────────────────────────────────
print(f"Expected regions ({len(EXPECTED_REGIONS)}):")
for i, r in enumerate(EXPECTED_REGIONS):
    end = '\n' if (i+1) % 6 == 0 else '  '
    print(f"  {r:<18}", end=end)
print()

# Quick extraction test
df_test = loader.extract_schools_region(2021)
found_regions = set(df_test['region'].unique())
missing = set(EXPECTED_REGIONS) - found_regions
extra   = found_regions - set(EXPECTED_REGIONS)
print(f"\nSchools extraction test (2021):")
print(f"  Rows extracted: {len(df_test)}")
print(f"  Regions found:  {len(found_regions)}")
print(f"  Missing:        {missing or 'None'}")
print(f"  Extra:          {extra or 'None'}")

In [ ]:
# ── Data availability summary visualisation ───────────────────────────────
fig, ax = plt.subplots(figsize=(14, 5))
avail_data = {
    'Schools by Region':       [1,1,1,1,1],
    'Enrolment (F1-F4)':       [1,1,1,1,1],
    'Teachers by Region':      [1,1,1,1,1],
    'CSEE Pass Rate (Nat)':    [1,1,1,1,1],
    'PTR / Teacher Summary':   [1,1,1,1,1],
    'Electricity Access':      [0,1,1,1,1],
    'ICT Equipment':           [0,1,1,1,1],
    'Dropout (National)':      [1,1,1,1,1],
    'Dropout by Region':       [1,1,1,1,1],
    'Completion Rate':         [1,1,1,1,1],
    'Textbook Ratio':          [0,1,1,1,1],
    'Disability Enrolment':    [0,1,1,1,1],
}
years = [2020, 2021, 2022, 2023, 2024]
mat = pd.DataFrame(avail_data, index=years).T
sns.heatmap(mat, ax=ax, cmap='RdYlGn', linewidths=0.5, linecolor='white',
            cbar=False, annot=True, fmt='d',
            annot_kws={'size':10},
            yticklabels=True, xticklabels=years)
ax.set_title('Data Availability Matrix -- BEST Secondary Education Indicators',
             fontweight='bold', pad=12)
ax.set_xlabel('Year')
plt.tight_layout()
plt.savefig(paths.fig('nb01_data_availability.png'), dpi=150, bbox_inches='tight')
plt.show()
print("\nGreen = data available, Red = not available / must be imputed")

In [ ]:
# ── Summary ──────────────────────────────────────────────────────────────
print("=" * 60)
print("DATA UNDERSTANDING SUMMARY")
print("=" * 60)
print(f"  Total workbooks loaded:    {len(loader.workbooks)}")
print(f"  Total sheets (all years):  {sum(len(xl.sheet_names) for xl in loader.workbooks.values())}")
print(f"  Analytical sheet mappings: {len(SHEET_MAP)}")
print(f"  Target regions:            {len(EXPECTED_REGIONS)}")
print()
print("  Key finding: 2020 uses a numbered flat-table format while 2021-2024")
print("  use named T3.* sheets. Sheet names also shift in 2024 (e.g. T3.25 -> T3.27).")
print("  Electricity and ICT data are not available for 2020 at regional level")
print("  and will be imputed via backward-fill in the cleaning pipeline.")
print()
print("  Next step: Notebook 02 -- Data Cleaning and Harmonisation")